In [2]:
# constants for LPG supply chain processing
default_crude2lpg = 0.025
default_FOB = 0.6  # USD/kg
STS_cost = 0.1  # 10%
pre_bottling_cost = 0.2  # 20% TO BE MODIFIED
CRUDE_BARREL_TO_TONS = 7.33  # barrel of crude per ton
VLGC_COMPLIANCE_RADIUS_KM = 20  # km radius for VLGC compliance check

import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np
from shapely.geometry import Point

# Set up data path
data_dir = Path("dataset_first_step")
data_dir.mkdir(exist_ok=True)

print("Constants declared and imports completed")

Constants declared and imports completed


In [3]:
# Load input data files
try:
    # Load GeoPackages
    refineries = gpd.read_file(data_dir / "refineries.gpkg")
    ports = gpd.read_file(data_dir / "ports.gpkg")
    primary_storage = gpd.read_file(data_dir / "primary_storage.gpkg")
    gasplants = gpd.read_file(data_dir / "gasplants.gpkg")
    
    # Load CSVs
    national_shares = pd.read_csv(data_dir / "national_shares.csv", index_col=0)
    lpg_import_cost = pd.read_csv(data_dir / "lpg_import_cost.csv", index_col=0)
    
    print("All input files loaded successfully")
    print(f"  - Refineries: {len(refineries)} records")
    print(f"  - Ports: {len(ports)} records")
    print(f"  - Gas plants: {len(gasplants)} records")
    print(f"  - Primary storage: {len(primary_storage)} records")
    
except FileNotFoundError as e:
    print(f"File not found: {e}")
    print("  Creating sample data instead...")

DataSourceError: dataset_first_step\refineries.gpkg: No such file or directory

In [ ]:
# Create sample gasplants.gpkg if it doesn't exist
if not (data_dir / "gasplants.gpkg").exists():
    # Sample gas plants data with 3 realistic elements
    gasplants_data = {
        'geometry': [
            Point(2.3522, 48.8566),    # Paris area
            Point(7.7497, 48.5734),    # Strasbourg area
            Point(-0.5596, 44.8378),   # Bordeaux area
        ],
        'name': ['Gas Plant North', 'Gas Plant East', 'Gas Plant West'],
        'LPG_capacity': [150.0, 200.0, 120.0],  # tons/year
        'price': [0.75, 0.72, 0.78],  # USD/kg
        'country': ['France', 'France', 'France']
    }
    
    gasplants = gpd.GeoDataFrame(gasplants_data, crs="EPSG:4326")
    gasplants.to_file(data_dir / "gasplants.gpkg", driver="GPKG")
    print("Sample gasplants.gpkg created with 3 elements")
else:
    gasplants = gpd.read_file(data_dir / "gasplants.gpkg")
    print("gasplants.gpkg loaded")

In [ ]:
# STEP 1: Process refineries - Calculate LPG capacity
def process_refineries(refineries_gdf, import_cost_df):
    """
    Add LPG_capacity field to refineries
    LPG_capacity = crude_capacity * crude2lpg * 365 / 7.33
    """
    refineries = refineries_gdf.copy()
    
    # Set default values for missing fields
    if 'crude_capacity' not in refineries.columns:
        refineries['crude_capacity'] = 0
    
    if 'crude2lpg' not in refineries.columns:
        refineries['crude2lpg'] = default_crude2lpg
    else:
        refineries['crude2lpg'] = refineries['crude2lpg'].fillna(default_crude2lpg)
    
    if 'price' not in refineries.columns:
        refineries['price'] = default_FOB * import_cost_df.loc['import_sea', 0]
    
    # Calculate LPG capacity: barrels/day * conversion factor * days/year / barrel-to-ton ratio
    refineries['LPG_capacity'] = (
        refineries['crude_capacity'] * 
        refineries['crude2lpg'] * 
        365 / 
        CRUDE_BARREL_TO_TONS
    )
    
    # Fill NaN values with 0
    refineries['LPG_capacity'] = refineries['LPG_capacity'].fillna(0)
    
    print(f"Refineries processed: {len(refineries)} records")
    print(f"  - LPG capacity range: {refineries['LPG_capacity'].min():.2f} - {refineries['LPG_capacity'].max():.2f} tons/year")
    
    return refineries

# Apply processing
refineries = process_refineries(refineries, lpg_import_cost)
refineries.to_file(data_dir / "refineries_processed.gpkg", driver="GPKG")

In [ ]:
# STEP 2: Process ports - Calculate tank capacity nearby, price, and VLGC compliance
def process_ports(ports_gdf, storage_gdf, import_cost_df):
    """
    For each port:
    1. Find tanks_nearby: total tank capacity within VLGC_COMPLIANCE_RADIUS_KM
    2. Calculate price based on FOB and import costs (add STS cost if not VLGC compliant)
    3. Check VLGC_compliance: true if at least one storage facility within 20km
    """
    ports = ports_gdf.copy()
    
    # Ensure all ports are in same CRS as storage
    ports = ports.to_crs(storage_gdf.crs)
    
    # Initialize new fields
    if 'VLGC_compliance' not in ports.columns:
        ports['VLGC_compliance'] = False
    
    ports['tanks_nearby'] = 0.0
    ports['LPG_compliance'] = False
    
    if 'FOB_price' not in ports.columns:
        ports['FOB_price'] = default_FOB
    else:
        ports['FOB_price'] = ports['FOB_price'].fillna(default_FOB)
    
    # Get import costs
    import_sea = import_cost_df.loc['import_sea', 0]
    
    # For each port, find nearby storage facilities
    for idx, port in ports.iterrows():
        port_geom = port.geometry
        
        # Find storage within radius
        distances = storage_gdf.geometry.distance(port_geom)
        nearby = storage_gdf[distances <= VLGC_COMPLIANCE_RADIUS_KM / 111.0]  # Convert km to degrees (approx)
        
        # Update tanks_nearby
        if 'tank_capacity' in storage_gdf.columns:
            ports.at[idx, 'tanks_nearby'] = nearby['tank_capacity'].sum()
        else:
            ports.at[idx, 'tanks_nearby'] = len(nearby)  # Count of facilities if no capacity field
        
        # Update LPG_compliance
        ports.at[idx, 'LPG_compliance'] = len(nearby) > 0
    
    # Calculate price
    ports['price'] = ports['FOB_price'] * import_sea
    
    # Add STS cost for non-VLGC compliant ports
    non_vlgc_mask = ports['VLGC_compliance'] == False
    ports.loc[non_vlgc_mask, 'price'] = ports.loc[non_vlgc_mask, 'price'] * (1 + STS_cost)
    
    if 'price' not in ports.columns:
        ports['price'] = default_FOB * import_sea
    
    print(f"Ports processed: {len(ports)} records")
    print(f"  - Ports with VLGC compliance: {ports['LPG_compliance'].sum()}")
    print(f"  - Price range: {ports['price'].min():.3f} - {ports['price'].max():.3f} USD/kg")
    
    return ports

# Apply processing
ports = process_ports(ports, primary_storage, lpg_import_cost)
ports.to_file(data_dir / "ports_processed.gpkg", driver="GPKG")

In [ ]:
# STEP 3: Process gas plants - Calculate LPG capacity
def process_gasplants(gasplants_gdf):
    """
    Ensure LPG_capacity and price fields exist with default values where needed
    """
    gasplants = gasplants_gdf.copy()
    
    if 'LPG_capacity' not in gasplants.columns:
        gasplants['LPG_capacity'] = 0
    else:
        gasplants['LPG_capacity'] = gasplants['LPG_capacity'].fillna(0)
    
    if 'price' not in gasplants.columns:
        gasplants['price'] = default_FOB
    else:
        gasplants['price'] = gasplants['price'].fillna(default_FOB)
    
    print(f"Gas plants processed: {len(gasplants)} records")
    print(f"  - LPG capacity range: {gasplants['LPG_capacity'].min():.2f} - {gasplants['LPG_capacity'].max():.2f} tons/year")
    
    return gasplants

# Apply processing
gasplants = process_gasplants(gasplants)
gasplants.to_file(data_dir / "gasplants_processed.gpkg", driver="GPKG")

In [ ]:
# STEP 4: Calculate percentages for each source by country
def calculate_percentages(refineries_gdf, ports_gdf, gasplants_gdf, national_shares_df, storage_gdf):
    """
    For each country:
    1. Get national % from national_shares CSV
    2. Calculate % of each facility on total capacity of that category
    3. Calculate relative % based on national share and facility share
    
    Returns a combined GeoDataFrame with all sources and their percentages
    """
    
    all_sources = []
    
    # Get unique countries (assuming 'country' field exists or use geometry-based country detection)
    countries = set()
    if 'country' in refineries_gdf.columns:
        countries.update(refineries_gdf['country'].unique())
    if 'country' in ports_gdf.columns:
        countries.update(ports_gdf['country'].unique())
    if 'country' in gasplants_gdf.columns:
        countries.update(gasplants_gdf['country'].unique())
    
    countries = list(countries)
    
    for country in countries:
        # Get national shares for this country
        if country in national_shares_df.index:
            nat_share_ref = national_shares_df.loc[country, 'refineries'] if 'refineries' in national_shares_df.columns else 0.33
            nat_share_port = national_shares_df.loc[country, 'ports'] if 'ports' in national_shares_df.columns else 0.33
            nat_share_gas = national_shares_df.loc[country, 'gasplants'] if 'gasplants' in national_shares_df.columns else 0.34
        else:
            nat_share_ref = nat_share_port = nat_share_gas = 1.0 / 3
        
        # Process refineries for this country
        ref_country = refineries_gdf[refineries_gdf['country'] == country] if 'country' in refineries_gdf.columns else refineries_gdf
        
        if len(ref_country) > 0:
            total_ref_capacity = ref_country['LPG_capacity'].sum()
            
            for idx, refinery in ref_country.iterrows():
                if total_ref_capacity > 0:
                    facility_share = refinery['LPG_capacity'] / total_ref_capacity
                else:
                    facility_share = 1.0 / len(ref_country)
                
                percentage = nat_share_ref * facility_share
                
                all_sources.append({
                    'geometry': refinery.geometry,
                    'name': refinery.get('name', f'Refinery_{idx}'),
                    'category': 'refineries',
                    'percentage': percentage,
                    'price': refinery.get('price', default_FOB),
                    'country': country
                })
        
        # Process ports for this country
        ports_country = ports_gdf[ports_gdf['country'] == country] if 'country' in ports_gdf.columns else ports_gdf
        
        if len(ports_country) > 0:
            # Use tank capacity if available, otherwise equal share
            total_port_capacity = ports_country['tanks_nearby'].sum()
            
            for idx, port in ports_country.iterrows():
                if total_port_capacity > 0:
                    facility_share = port['tanks_nearby'] / total_port_capacity
                else:
                    facility_share = 1.0 / len(ports_country)
                
                percentage = nat_share_port * facility_share
                
                all_sources.append({
                    'geometry': port.geometry,
                    'name': port.get('name', f'Port_{idx}'),
                    'category': 'ports',
                    'percentage': percentage,
                    'price': port.get('price', default_FOB),
                    'country': country
                })
        
        # Process gas plants for this country
        gas_country = gasplants_gdf[gasplants_gdf['country'] == country] if 'country' in gasplants_gdf.columns else gasplants_gdf
        
        if len(gas_country) > 0:
            total_gas_capacity = gas_country['LPG_capacity'].sum()
            
            for idx, plant in gas_country.iterrows():
                if total_gas_capacity > 0:
                    facility_share = plant['LPG_capacity'] / total_gas_capacity
                else:
                    facility_share = 1.0 / len(gas_country)
                
                percentage = nat_share_gas * facility_share
                
                all_sources.append({
                    'geometry': plant.geometry,
                    'name': plant.get('name', f'GasPlant_{idx}'),
                    'category': 'gasplants',
                    'percentage': percentage,
                    'price': plant.get('price', default_FOB),
                    'country': country
                })
    
    # Create GeoDataFrame
    result_gdf = gpd.GeoDataFrame(all_sources, crs=refineries_gdf.crs)
    
    print(f"Percentages calculated: {len(result_gdf)} total sources")
    print(f"  - Refineries: {len(result_gdf[result_gdf['category']=='refineries'])}")
    print(f"  - Ports: {len(result_gdf[result_gdf['category']=='ports'])}")
    print(f"  - Gas plants: {len(result_gdf[result_gdf['category']=='gasplants'])}")
    print(f"  - Total percentage: {result_gdf['percentage'].sum():.2%}")
    
    return result_gdf

# Apply processing
beginning = calculate_percentages(refineries, ports, gasplants, national_shares, primary_storage)

In [ ]:
# STEP 5: Save beginning.gpkg with final output
def save_beginning_gpkg(beginning_gdf, output_path):
    """
    Save the beginning GeoDataFrame with fields: name | category | percentage | price
    """
    # Select and order the columns as specified
    output_gdf = beginning_gdf[['name', 'category', 'percentage', 'price', 'geometry']].copy()
    
    # Save to GeoPackage
    output_gdf.to_file(output_path, driver="GPKG")
    
    print(f"beginning.gpkg saved to {output_path}")
    print(f"  - Format: name | category | percentage | price")
    print(f"  - Total records: {len(output_gdf)}")
    print("\nSample records:")
    print(output_gdf[['name', 'category', 'percentage', 'price']].head(10).to_string())
    
    return output_gdf

# Save final output
beginning_final = save_beginning_gpkg(beginning, data_dir / "beginning.gpkg")

In [ ]:
# STEP 6: Summary and Verification
print("="*70)
print("LPG SUPPLY CHAIN PROCESSING - SUMMARY")
print("="*70)

print(f"\nCONSTANTS USED:")
print(f"  - default_crude2lpg: {default_crude2lpg}")
print(f"  - default_FOB: {default_FOB} USD/kg")
print(f"  - STS_cost: {STS_cost} (10%)")
print(f"  - pre_bottling_cost: {pre_bottling_cost} (20%)")
print(f"  - VLGC_compliance_radius: {VLGC_COMPLIANCE_RADIUS_KM} km")

print(f"\nDATA PROCESSING RESULTS:")
print(f"  - Total supply sources: {len(beginning_final)}")

# Group by category
category_summary = beginning_final.groupby('category').agg({
    'percentage': 'sum',
    'name': 'count'
}).rename(columns={'name': 'count'})

print(f"\n  By category:")
for cat in category_summary.index:
    count = int(category_summary.loc[cat, 'count'])
    pct = category_summary.loc[cat, 'percentage']
    print(f"    - {cat}: {count} sources, {pct:.2%} total share")

# Group by country
if 'country' in beginning_final.columns:
    country_summary = beginning_final.groupby('country').agg({
        'percentage': 'sum',
        'name': 'count'
    }).rename(columns={'name': 'count'})
    
    print(f"\n  By country:")
    for country in country_summary.index:
        count = int(country_summary.loc[country, 'count'])
        pct = country_summary.loc[country, 'percentage']
        print(f"    - {country}: {count} sources, {pct:.2%} total share")

print(f"\nOUTPUT FILES SAVED in: {data_dir}")
print(f"  - beginning.gpkg (main output)")
print(f"  - refineries_processed.gpkg")
print(f"  - ports_processed.gpkg")
print(f"  - gasplants_processed.gpkg")

print(f"\nProcessing complete!")
print("="*70)